# 06. 모델 후보 비교 (시간 분리 검증)

05_final_submission(LR + 규칙 rank 평균, 본선 1회차 리더보드 0.4554)을 기준선으로, 모델/가중치 후보를 **같은 조건**에서 비교한다.

- 검증: 학습 07-26~07-30 (FUT=26 자체 라벨) → 평가 공식 08-25 (라벨 창이 겹치지 않는 유일한 시간 분리)
- 04_final.ipynb의 leave-one-date-out CV는 학습에 나중 기준일이 들어가 **시간 누수**가 있어, 거기서 LGB가 좋아 보인 결과는 이 노트북 결과로 대체한다.
- 평가 기준일이 08-25 하나(666행)라 0.01~0.02 차이는 노이즈로 본다.
- 환경: 05와 동일 (피처/라벨/헬퍼 셀은 05에서 그대로 복사)

In [1]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW = '../data/raw/epoch_data'
D = '2026-09-01'          # 본선 기준일
FUT = 26                  # 본선 정답 구간 길이 (일)
SEED = 42

con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

,min(published_at),max(published_at),count_star()
0,2026-07-10 08:39:51,2026-08-31 23:59:27,31524


In [2]:
def make_features(Dref):
    f = con.sql(f"""
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,
               MEDIAN(view_5d)                                 AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,
               STDDEV(LN(1 + view_5d))                         AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,
               MIN(days_ago)                                   AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UCYG8tuvbQzWsDeoADyJFU8Q,8.123558,28,0.018834,8,0,0.000000,1.250000,0.109680,0.377461,0,0,0.530436,0.127477,0.001020,7.989204,8.287556,0.015644,1.250000,4,19,19006.470588,NaN,3.367296
1,UCSh__r5HS4Y1xBkccWnV0TA,12.094163,21,0.024374,7,0,0.000000,0.250000,-0.286160,0.573059,1,1,0.128447,-0.330043,0.000889,11.355371,12.625628,-0.003870,0.250000,4,14,1926.920000,NaN,3.091042
2,UCTNyl3P7q6PNhPddh6d7WJQ,3.135494,8,0.066316,2,4,0.666667,1.083333,7.214280,3.136847,1,1,7.214280,7.577186,0.000000,2.458803,8.580298,0.111421,1.083333,2,4,2631.250000,4.596217,2.197225
3,UCQzgMwT5Q6x-Mtq65HRS_0Q,8.667164,23,0.007524,8,2,0.200000,0.541667,0.525929,2.811501,0,0,0.525929,0.905312,0.000924,5.610890,9.943376,0.046821,0.541667,4,16,1330.259259,0.341978,3.178054
4,UCg80LjrsnWo0T3T3uYCPhrQ,9.383537,3,0.011102,0,0,NaN,24.625000,NaN,1.213276,0,0,NaN,NaN,0.016821,8.848230,10.731866,-0.174413,24.625000,0,2,2478.666667,NaN,1.386294


In [3]:
def make_labels(Dref, fut=FUT):
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        scored AS (
          SELECT channel_id, LN(1 + future_score) - LN(1 + past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        SELECT channel_id,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    """).df()
    g['ref_date'] = Dref
    return g


def attach_features(labels):
    return pd.concat([labels[labels.ref_date == d].merge(make_features(d), on='channel_id', how='left')
                      for d in sorted(labels.ref_date.unique())], ignore_index=True)


# 우리가 만든 26일 라벨 — 과거 이력이 2주 이상 쌓인 07-26 ~ 08-05
OWN_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()
own = attach_features(pd.concat([make_labels(d) for d in OWN_DATES], ignore_index=True))

# 공식 라벨 (18일 창)
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
tl['ref_date']   = tl.row_id.str.rsplit('_', n=1).str[1]
official = attach_features(tl[['channel_id', 'ref_date', 'target']])

summary = pd.concat([own, official]).groupby('ref_date').target.agg(['size', 'mean']).round(3)
summary['window'] = ['26일(자체)'] * len(OWN_DATES) + ['18일(공식)'] * 2
summary

,size,mean,window
ref_date,,,
2026-07-26,349,0.201,26일(자체)
2026-07-27,389,0.201,26일(자체)
2026-07-28,414,0.200,26일(자체)
2026-07-29,430,0.200,26일(자체)
2026-07-30,448,0.201,26일(자체)
2026-07-31,479,0.200,26일(자체)
2026-08-01,494,0.200,26일(자체)
2026-08-02,509,0.200,26일(자체)
2026-08-03,523,0.201,26일(자체)


In [4]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2



In [5]:
import lightgbm as lgb

F = FEATURES
tr = own[own.ref_date <= '2026-07-30']
va = official[official.ref_date == '2026-08-25']
y  = va.target
ap = average_precision_score

rule = rank01(-va.log_past_med)
p_lr = logreg(0.1).fit(tr[F], tr.target).predict_proba(va[F])[:, 1]

def gb(seed, **kw):
    p = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=3, min_child_samples=60,
             subsample=0.7, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10, random_state=seed, verbose=-1)
    p.update(kw)
    return lgb.LGBMClassifier(**p)

p_gb  = np.mean([gb(s).fit(tr[F], tr.target).predict_proba(va[F])[:, 1] for s in range(5)], axis=0)
p_gb2 = np.mean([gb(s, n_estimators=100).fit(tr[F], tr.target).predict_proba(va[F])[:, 1] for s in range(5)], axis=0)

rows = [
    ('규칙 -log_past_med',            rule),
    ('LR (C=0.1)',                    p_lr),
    ('LGB (200트리, 5시드)',           p_gb),
    ('LGB (100트리, 5시드)',           p_gb2),
    ('LR + 규칙 (현재 제출)',          rank01(p_lr) + rule),
    ('LGB + 규칙',                    rank01(p_gb) + rule),
    ('LR + LGB',                      rank01(p_lr) + rank01(p_gb)),
    ('LR + LGB + 규칙',               rank01(p_lr) + rank01(p_gb) + rule),
    ('LR + LGB + 규칙x2',             rank01(p_lr) + rank01(p_gb) + 2 * rule),
]
for w in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    rows.append((f'LR*(1-w)+규칙*w, w={w}', (1 - w) * rank01(p_lr) + w * rule))

res = pd.DataFrame([(n, round(ap(y, s), 4)) for n, s in rows], columns=['방법', 'PR-AUC(08-25)'])
res

,방법,PR-AUC(08-25)
0,규칙 -log_past_med,0.4039
1,LR (C=0.1),0.3815
2,"LGB (200트리, 5시드)",0.3370
3,"LGB (100트리, 5시드)",0.3405
4,LR + 규칙 (현재 제출),0.4299
5,LGB + 규칙,0.4112
6,LR + LGB,0.3619
7,LR + LGB + 규칙,0.4186
8,LR + LGB + 규칙x2,0.4300
9,"LR*(1-w)+규칙*w, w=0.3",0.4204


## 결과 해석
- 규칙(−log_past_med) 0.404, LR 0.382, **LR+규칙 0.430** (현재 제출). LGB는 0.337로 LR보다 뚜렷이 낮고, blend에 넣어도 이득이 없다 → 트리 모델은 과적합.
- 규칙 가중치 0.5~0.6 부근이 평평해 더 조정할 이득이 없다.
- 결론: 현재 제출(LR + 규칙 0.5)을 기준선으로 유지. 더 올리려면 모델이 아니라 신호(과거 규칙 개선, 미래 업로드 유무 분해 등) 쪽을 시험해야 한다.